# Tutorial: Train and Evaluator your Own Classifier

### Objective
To train a binary classifier that differentiates tumours from simple cysts.

### Prerequisites:

- Any dataset with 3D images and annotations masks. In this example we us the [KiTS23](https://github.com/neheller/kits23) data.
- RenalVision


## 1. Dataset Preparation
First we need to have an overview of our data. If you downloaded the KITS data with the official toolkit you should have a folder with 588 cases and a kits23.json. Each case consists of a file `imaging.nii.gz` and `segmentation.nii.gz`. Lets store the paths to these in a `dataset.csv` file:

In [ ]:
import os
from pathlib import Path

import pandas as pd

# dataset_path = Path("/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/kidney/temp_data/kits23/dataset")
dataset_path = Path("kits23/dataset")
cases = [c for c in os.listdir(dataset_path) if "case" in c]

# filter for cases that we have images for (in case you didn't download the whole dataset)
valid_cases = []
for c in cases:
    if (dataset_path / c / "imaging.nii.gz").exists():
        valid_cases.append(c)
print(f"Found data for {len(valid_cases)} cases")

# Create a Dataframe
data = pd.DataFrame()
data["case"] = valid_cases
data["image_path"] = data["case"].apply(lambda c: str(dataset_path / c / "imaging.nii.gz"))
data["seg_path"] = data["case"].apply(lambda c: str(dataset_path / c / "segmentation.nii.gz"))

data.to_csv("dataset.csv",index=False)

Next, we need to make sure that the segmentation masks in our dataset have unique labels for each target class that we want to classify. In the KiTS data we have the classes (1) Kidney, (2) Tumour, and (3) Cyst. This is one label too much, so we need to exclude the kidney annotations. Luckily RenalVision can handle this automatically if we specifiy a custom class mapping:

In [ ]:
import json

labelmap = {
    1:0, # exclude kidney
    2:1, # First class: tumours
    3:2, # Second class: cysts
}

with open("labelmap.json","w") as f:
    json.dump(labelmap,f)

## 2. Feature Extraction
We want to save an embedding for each masked component. First we need to choose a backbone for calculating these embeddings: Here, we decide on the `radiomics` configuration, which is lightweight and captures a lot of information. `rv extract` now calculates an embedding for each masked region and saves it together with the orginal segmentation class. 

This is a one-time operation and may take a few minutes. You can increase speed by setting `--cores_per_job`. Tipp: if you have sufficient RAM increase `--num_jobs` instead and adjust cores_per_job accordingly. For the KITS data we recommend ~10GB RAM per job. (For the CTFM and FMCIB foundation models num_jobs needs to be 1)

In [ ]:
! rv extract \
    --data ../data/KITS.csv \
    --extractor radiomics \
    --output radiomics.parquet \
    --label-map labelmap.json \
    --num_jobs 2 \
    --cores_per_job 6 


features = pd.read_parquet("radiomics.parquet")

print(f"Extracted {len(features)} features in total.")
print("Number of features per class:", features["class_id"].value_counts())

As you can see some images have multiple cysts or tumours, so in total we get more embeddings than we had images.

## 2. Training and Evaluation
Now that we have embeddings we can train a classifier:

In [ ]:
from renal_vision.shared.utils import generate_patient_fold_mapping

# First, lets split data in train and validation folds
features = pd.read_parquet("radiomics.parquet")
fold_map = generate_patient_fold_mapping(features,group_col="case", stratify_col="class_id", n_folds=5)
features["fold"] = features["case"].astype(str).map(fold_map)

val_fold = 0
val = features[features["fold"] == val_fold]
train = features[features["fold"] != val_fold]

# quality check
classes = features["class_id"].unique()
for cl in classes:
    if cl not in train["class_id"].values:
        print("WARNING:", f"Class {cl} not in training data")
    if cl not in val["class_id"].values:
        print("WARNING:", f"Class {cl} not in validation data")

# save
val.to_parquet("val_data.parquet", index=False)
train.to_parquet("train_data.parquet", index=False)

Lets train a classifer, xgboost is generally best for that.
The configuration file specifiying the preprocessing steps was automatically created during feature extraction.


In [ ]:
! rv train \
    --data train_data.parquet \
    --extractor-config radiomics.config.json \
    --model xgboost \
    --output-dir ./model

Now lets evaluate our results. 
(You need to run this in a dedicated terminal, because IPython doesnt like our plotting logic.)

```bash
rv eval \
    --data val_data.parquet \
    --model ./model/model.pkl \
    --output-dir ./model
```